# Baseline Evaluation — qwen3_8b

Text-only zero-shot evaluation on the GLOBEM BDI dataset.
Predicts BDI severity category (Minimal / Mild / Moderate / Severe) from serialized behavioral prompts.

**Requirements:** `transformers`, `accelerate`, `huggingface_hub`, `pandas`  
**GPU:** Recommended (A10G / T4 or better). Falls back to CPU but will be slow.

In [ ]:
import json
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# ── Add notebooks/ dir to path so eval_utils is importable ───────────────────
sys.path.insert(0, str(Path.cwd()))
from eval_utils_bdi import slugify, normalize_text, extract_bdi_prediction, compute_metrics
import warnings
import transformers
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

ROOT        = Path.cwd().parent
CONFIG_PATH = ROOT / "config" / "eval_config.json"
MODEL_KEY   = "qwen3_8b"

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config = json.load(f)

model_name         = config["models"][MODEL_KEY]
system_prompt_text = config["system_prompt_text"]
system_prompt_id   = slugify(system_prompt_text)
model_slug         = slugify(MODEL_KEY)

print(f"Model key   : {MODEL_KEY}")
print(f"Model name  : {model_name}")
print(f"Prompt id   : {system_prompt_id}")
print(f"Device      : {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Authenticate and load model

In [ ]:
# Qwen3 models are gated on HuggingFace — log in with your HF token.
# Run `login()` once; it caches your token in ~/.cache/huggingface/.
from huggingface_hub import login
import os
# Can create a separate cell and run: os.environ["HF_TOKEN"] = "token here"
login(token=os.environ["HF_TOKEN"])

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

pipe = pipeline(
    "text-generation",          # text-only; no image column used
    model=model_name,
    torch_dtype=dtype,
    device_map="auto",          # uses GPU if available, else CPU
)
print(f"Pipeline ready: {pipe.model.__class__.__name__}")

## 2. Load dataset

In [ ]:
data_csv = ROOT / config["data_csv"]
df = pd.read_csv(data_csv)

# Validate required columns exist (image_path_or_url is present but unused)
required_cols = set(config["required_columns"])
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# gold_answer must be one of the four BDI severity categories
valid_labels = {"Minimal", "Mild", "Moderate", "Severe"}
bad_labels = df[~df["gold_answer"].astype(str).isin(valid_labels)]
if len(bad_labels):
    raise ValueError(f"{len(bad_labels)} rows have unexpected gold_answer values.")

print(f"Loaded {len(df):,} rows from {data_csv.name}")
print(f"Label distribution:\n{df['gold_answer'].value_counts().sort_index()}")

## 3. Run inference

In [ ]:
results   = []
run_ts    = datetime.now().strftime("%Y%m%d_%H%M%S")
gen_cfg   = config.get("generation", {"max_new_tokens": 10, "do_sample": False})

for _, row in df.iterrows():
    sample_id   = row["sample_id"]
    question    = str(row["question"])
    gold_answer = str(row["gold_answer"]).strip()

    # Build a chat-style message list (no image content)
    messages = [
        {"role": "system",  "content": system_prompt_text},
        {"role": "user",    "content": question},
    ]

    output_obj = pipe(messages, **gen_cfg)
    pred_text  = extract_bdi_prediction(output_obj)

    results.append({
        "sample_id":          sample_id,
        "model_name":         model_name,
        "system_prompt_id":   system_prompt_id,
        "system_prompt_text": system_prompt_text,
        "question":           question,
        "gold_answer":        gold_answer,
        "prediction_raw":     json.dumps(output_obj, ensure_ascii=False),
        "prediction_text":    pred_text,
        "exact_match":        int(pred_text == gold_answer),
        "normalized_match":   int(normalize_text(pred_text) == normalize_text(gold_answer)),
        "timestamp":          datetime.now().isoformat(),
    })
    if len(results) % 50 == 0:
        print(f"{len(results)}/{len(df)} done", flush=True)

results_df = pd.DataFrame(results)
print(f"Inference complete. Parsed predictions: "
      f"{results_df['prediction_text'].ne('').sum():,} / {len(results_df):,}")
results_df[["sample_id", "gold_answer", "prediction_text", "exact_match"]].head(5)

## 4. Compute metrics

In [ ]:
metrics_vals = compute_metrics(results)
metrics = {
    "model_name":         model_name,
    "model_slug":         model_slug,
    "system_prompt_id":   system_prompt_id,
    "system_prompt_text": system_prompt_text,
    "created_at":         datetime.now().isoformat(),
    **metrics_vals,
}

print(json.dumps(metrics, indent=2))

## 5. Save results

In [ ]:
results_dir = ROOT / config["results_dir"]
results_dir.mkdir(parents=True, exist_ok=True)

stem             = f"globem_{model_slug}_{system_prompt_id}_{run_ts}"
predictions_path = results_dir / f"{stem}_predictions.csv"
metrics_path     = results_dir / f"{stem}_metrics.json"

results_df.to_csv(predictions_path, index=False)
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f"Predictions → {predictions_path}")
print(f"Metrics     → {metrics_path}")

In [ ]:
import torch, gc

del pipe
gc.collect()
torch.cuda.empty_cache()